# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score


## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.

In [2]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df_days = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = df_days["dayofweek"].values

X = df.drop(columns="dayofweek")
y = df["dayofweek"]

2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.

In [4]:
svc_best_params = {
    "C": 10,
    "class_weight": None,
    "gamma": "auto",
    "kernel": "rbf",
    "probability": True,
    "random_state": 21,
}

svc_model = SVC(**svc_best_params)
svc_model.fit(X_train, y_train)
y_pred = svc_model.predict(X_test)

2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [5]:
acc = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test, y_pred, average="weighted"
)  # average="weighted" рассчитывает метрику для каждого класса, но при усреднении учитывает реальный объем (пропорцию) каждого класса в тестовой выборке y_test
recall = recall_score(y_test, y_pred, average="weighted")

# ROC AUC: Receiver Operating Characteristic – Area Under Curve - Площадь под кривой зависимостей True Positive Rate от False Positive Rate при варьировании порога классификации

# Суть метрики коротко: она показывает вероятность того, что случайно выбранный положительный объект получит от модели оценку выше, чем случайно выбранный отрицательный объект. Значение меняется от 0 до 1. Чем ближе к 1, тем лучше модель разделяет классы.

# Для вычисления ROC AUC критически важно передавать именно вероятности (y_prob), а не предсказанные классы (y_pred).
y_prob = svc_model.predict_proba(X_test)

# по ТЗ multi_class='ovo' One-vs-One (все возможные попарные комбинации классов)
roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovo", average="weighted")

In [6]:
print(f"accuracy is {acc:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [7]:
dt_best_params = {
    "class_weight": "balanced",
    "criterion": "gini",
    "max_depth": 21,
    "random_state": 21,
}

dt_model = DecisionTreeClassifier(**dt_best_params)
dt_model.fit(X_train, y_train)
y_predDT = dt_model.predict(X_test)

In [8]:
acc_dt = accuracy_score(y_test, y_predDT)
precision_dt = precision_score(y_test, y_predDT, average="weighted")
recall_dt = recall_score(y_test, y_predDT, average="weighted")
y_prob_dt = dt_model.predict_proba(X_test)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt, multi_class="ovo", average="weighted")

In [9]:
print(f"accuracy is {acc_dt:.5f}")
print(f"precision is {precision_dt:.5f}")
print(f"recall is {recall_dt:.5f}")
print(f"roc_auc is {roc_auc_dt:.5f}")

accuracy is 0.88462
precision is 0.88765
recall is 0.88462
roc_auc is 0.93528


## 4. Random forest

1. The same task for random forest.

In [10]:
rf_best_params = {
    "class_weight": "balanced",
    "criterion": "entropy",
    "max_depth": 24,
    "n_estimators": 100,
    "random_state": 21,
}

rf_model = RandomForestClassifier(**rf_best_params)
rf_model.fit(X_train, y_train)
y_predRF = rf_model.predict(X_test)

In [11]:
acc_rf = accuracy_score(y_test, y_predRF)
precision_rf = precision_score(y_test, y_predRF, average="weighted")
recall_rf = recall_score(y_test, y_predRF, average="weighted")
y_prob_rf = rf_model.predict_proba(X_test)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf, multi_class="ovo", average="weighted")

In [12]:
print(f"accuracy is {acc_rf:.5f}")
print(f"precision is {precision_rf:.5f}")
print(f"recall is {recall_rf:.5f}")
print(f"roc_auc is {roc_auc_rf:.5f}")

accuracy is 0.92604
precision is 0.92754
recall is 0.92604
roc_auc is 0.98939


In [13]:
summary_df = pd.DataFrame(
    {
        "Model": ["svc_model", "dt_mode", "rf_model"],
        "accuracy": [acc, acc_dt, acc_rf],
        "precision": [precision, precision_dt, precision_rf],
        "recall": [recall, recall_dt, recall_rf],
        "roc auc": [roc_auc, roc_auc_dt, roc_auc_rf],
        "params": [svc_model, dt_model, rf_model],
    }
)
(
    summary_df.drop("params", axis=1)
    .style.highlight_max(
        color="lightgreen",
        axis=0,
    )
    .format(
        {
            "accuracy": "{:.5f}",
            "precision": "{:.5f}",
            "recall": "{:.5f}",
            "roc auc": "{:.5f}",
        }
    )
    .hide_index()
)


Model,accuracy,precision,recall,roc auc
svc_model,0.88757,0.89267,0.88757,0.97878
dt_mode,0.88462,0.88765,0.88462,0.93528
rf_model,0.92604,0.92754,0.92604,0.98939


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [15]:
summary_df.loc[2, "params"]

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=24, random_state=21)

In [73]:
test_df = X_test.copy()
test_df["y_test"] = y_test
test_df["y_pred"] = pd.Series(y_predRF, index=y_test.index)
test_errors_df = test_df[test_df["y_test"] != test_df["y_pred"]]
# test_errors_df.head(3)

In [74]:
# фильтруем только колонки юзеров
test_errors_df.filter(like="uid_user_").head(3);

In [75]:
# кол-во ошибок для каждого юзера
user_errors_data = test_errors_df.filter(like="uid_user_").sum(axis=0)
# user_errors_data.head(3)

In [44]:
# общее количество строк для каждого юзера во всем датасете
all_users_data = X.filter(like="uid_user_").sum(axis=0)

In [76]:
user_info = user_errors_data / all_users_data * 100
# user_info.head(3)

In [ ]:
lab_errors_data = test_errors_df.filter(like="labname_").sum(axis=0)

In [ ]:
all_labs_data = X.filter(like="labname_").sum(axis=0)

In [78]:
lab_info = lab_errors_data / all_labs_data * 100
# lab_df.head(3)

In [81]:
worst_user = user_info.idxmax()
worst_user_val = user_info.max()

In [82]:
worst_lab = lab_info.idxmax()
worst_lab_val = lab_info.max()

In [ ]:
weekday_errors = test_errors_df["y_test"].value_counts()
weekday_all = df["dayofweek"].value_counts()
weekday_info = weekday_errors / weekday_all * 100

In [86]:
days_dict = {0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"}
weekday_info.index = weekday_info.index.map(days_dict)
worst_day = weekday_info.idxmax()
worst_day_val = weekday_info.max()

In [87]:
print(
    f"The model makes the most errors for weekday: {worst_day} ({worst_day_val:.2f}%)"
)
print(f"Most errors for user: {worst_user} ({worst_user_val:.2f}%)")
print(f"Most errors for lab:  {worst_lab} ({worst_lab_val:.2f}%)")

The model makes the most errors for weekday: nan (4.41%)
Most errors for user: uid_user_6 (16.67%)
Most errors for lab:  labname_lab03 (100.00%)


## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.